# ImageNet-1K formal training: VisionLLaMA + LSSO/RRLSSO

Google Colab / Colab Enterprise entry point for one RTX PRO 6000 Blackwell 96GB. Run cells from top to bottom. The notebook clones the requested branch, installs and verifies the SM120 MathDx backend, detects the largest local scratch disk, validates gated ImageNet access, runs a real smoke test, and only then launches resumable training.

## 1. Get the repository
Edit only `REPO_URL` if your remote differs, then click the cell. The branch must already be pushed to the remote.

In [ ]:
from pathlib import Path
import os, sys, subprocess, getpass, json, shutil, signal, time

REPO_URL = 'https://github.com/Yang916-yy/LSSO.git'
BRANCH = 'experiment/vision-llama-bidirectional'
COLAB_ROOT = Path('/content') if Path('/content').is_dir() else Path.home()
ROOT = COLAB_ROOT / 'LSSO'
if not (ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
os.chdir(ROOT)
print('repository:', ROOT)
print('commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
# Keep PyTorch on the same CUDA major/minor as Colab's nvcc. MathDx uses
# device LTO, so CUDA 12 and CUDA 13 build components cannot be mixed.
nvcc = shutil.which('nvcc')
assert nvcc, 'The CUDA toolkit/nvcc is missing from this runtime'
nvcc_version = subprocess.check_output([nvcc, '--version'], text=True)
torch_cuda = subprocess.check_output(
    [sys.executable, '-c', 'import torch; print(torch.version.cuda or \"none\")'], text=True
).strip()
if 'release 12.8' in nvcc_version:
    wanted_cuda = '12.8'
    torch_packages = ('torch==2.11.0', 'torchvision==0.26.0')
    torch_index = 'https://download.pytorch.org/whl/cu128'
elif 'release 13.' in nvcc_version:
    wanted_cuda = '13.0'
    torch_packages = ('torch==2.12.1', 'torchvision==0.27.1')
    torch_index = 'https://download.pytorch.org/whl/cu130'
else:
    raise RuntimeError(f'Expected CUDA Toolkit 12.8 or 13.x, got:\n{nvcc_version}')
if not torch_cuda.startswith(wanted_cuda):
    assert 'torch' not in sys.modules, 'Restart the runtime, then run from the first cell'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                    *torch_packages, '--index-url', torch_index], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[experiments]'], cwd=ROOT, check=True)
print('dependencies installed; PyTorch CUDA:',
      subprocess.check_output([sys.executable, '-c', 'import torch; print(torch.version.cuda)'], text=True).strip())

## 2. Authenticate and choose storage
Accept the ImageNet terms first. The token is entered without echo and is never written to the notebook, config, log, or checkpoint.

In [ ]:
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face read token: ')
assert os.environ['HF_TOKEN'].startswith('hf_'), 'Expected a Hugging Face token'
print('token loaded for this runtime only')

In [ ]:
# Prefer the writable local filesystem with the most free space. Colab can
# expose large read-only mounts such as /kaggle/input, so df alone is unsafe.
preferred = [Path('/mnt/local-scratch'), Path('/content'), Path('/local_nvme'), Path('/tmp'), Path.home()]
df_lines = subprocess.check_output(['df', '-P', '-B1'], text=True).splitlines()[1:]
preferred.extend(Path(parts[5]) for line in df_lines if len(parts := line.split()) >= 6)
writable_disks, seen_devices = [], set()
for candidate in preferred:
    if str(candidate).startswith(('/content/drive', '/kaggle/input', '/proc', '/sys', '/dev')):
        continue
    try:
        if not candidate.is_dir():
            continue
        device = candidate.stat().st_dev
        if device in seen_devices:
            continue
        probe = candidate / f'.lsso-write-probe-{os.getpid()}'
        probe.mkdir(exist_ok=False)
        probe.rmdir()
        writable_disks.append((shutil.disk_usage(candidate).free, candidate))
        seen_devices.add(device)
    except (OSError, PermissionError):
        continue
assert writable_disks, 'No writable local filesystem was found'
SCRATCH_ROOT = max(writable_disks, key=lambda item: item[0])[1]
CACHE_ROOT = SCRATCH_ROOT / 'lsso-imagenet-wds'
OUTPUT_ROOT = ROOT / 'runs' / 'imagenet1k'
CONFIG = {
    'model': 'vision_llama_base_rrlsso_r32',  # change to vision_llama_base_lsso_r32 for the paired run
    'cache_dir': str(CACHE_ROOT),
    'output': str(OUTPUT_ROOT / 'vision_llama_base_rrlsso_r32'),
    'epochs': 300,
    'batch_size': 768,
    'eval_batch_size': 768,
    'grad_accum': 1,
    'workers': 32,
    'eval_workers': 4,  # independent, non-persistent validation pool
    'seed': 0,
}
Path(CONFIG['cache_dir']).mkdir(parents=True, exist_ok=True)
Path(CONFIG['output']).mkdir(parents=True, exist_ok=True)
print('selected scratch:', SCRATCH_ROOT)
print(json.dumps(CONFIG, indent=2))

## 3. Checks
Check CUDA, model registration, disk space, and gated repository access before spending GPU time.

In [ ]:
import torch, timm
import examples.models
from huggingface_hub import HfApi

assert torch.cuda.is_available(), 'CUDA is unavailable'
assert CONFIG['model'] in timm.list_models('vision_llama_*')
HfApi(token=os.environ['HF_TOKEN']).dataset_info('timm/imagenet-1k-wds')
disk = os.statvfs(CONFIG['cache_dir'])
free_gb = disk.f_bavail * disk.f_frsize / 2**30
assert free_gb >= 180, f'Only {free_gb:.1f} GiB free; reserve at least 180 GiB'
gpu_name = torch.cuda.get_device_name()
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
compute_capability = torch.cuda.get_device_capability()
assert 'RTX PRO 6000' in gpu_name and compute_capability == (12, 0) and gpu_gb >= 90, (
    f'Expected RTX PRO 6000 Blackwell 96GB (SM120), got {gpu_name}, ' 
    f'SM{compute_capability[0]}{compute_capability[1]} ({gpu_gb:.1f} GiB)'
)
assert torch.version.cuda and torch.version.cuda.startswith(wanted_cuda), (
    f'Expected PyTorch CUDA {wanted_cuda}, got {torch.version.cuda}'
)
print(gpu_name, f'SM{compute_capability[0]}{compute_capability[1]},',
      f'{gpu_gb:.1f} GiB VRAM, {free_gb:.1f} GiB cache space free')

### Build the fused SM120 backend
This downloads NVIDIA MathDx once per Colab runtime, compiles only for the allocated GPU, and fails loudly if the optimized backend cannot be loaded.

In [ ]:
import tarfile, urllib.request

if 'release 12.8' in nvcc_version:
    CUDA_PACKAGE, MATHDX_VERSION = 'cuda12', '25.12.1'
elif 'release 13.' in nvcc_version:
    CUDA_PACKAGE, MATHDX_VERSION = 'cuda13', '26.06.0'
else:
    raise RuntimeError(nvcc_version)
MATHDX_PACKAGE = f'nvidia-mathdx-{MATHDX_VERSION}-{CUDA_PACKAGE}'
MATHDX_CACHE = COLAB_ROOT / '.cache' / 'lsso-mathdx' / MATHDX_PACKAGE
MATHDX_ARCHIVE = MATHDX_CACHE / f'{MATHDX_PACKAGE}.tar.gz'
MATHDX_CACHE.mkdir(parents=True, exist_ok=True)
configs = list(MATHDX_CACHE.rglob('mathdx-config.cmake'))
if not configs:
    url = (
        'https://developer.nvidia.com/downloads/compute/cublasdx/redist/'
        f'cublasdx/{CUDA_PACKAGE}/{MATHDX_PACKAGE}.tar.gz'
    )
    if not MATHDX_ARCHIVE.is_file():
        print('downloading', url)
        urllib.request.urlretrieve(url, MATHDX_ARCHIVE)
    with tarfile.open(MATHDX_ARCHIVE, 'r:gz') as archive:
        archive.extractall(MATHDX_CACHE, filter='data')
    configs = list(MATHDX_CACHE.rglob('mathdx-config.cmake'))
assert len(configs) == 1, f'Expected one MathDx CMake config, found {configs}'
mathdx_root = configs[0].parents[3]
cuda_root = Path(nvcc).resolve().parents[1]
build_env = os.environ.copy()
build_env.update({
    'PYTHON_BIN': sys.executable,
    'MATHDX_ROOT': str(mathdx_root),
    'CUDA_HOME': str(cuda_root),
    'LSSO_CUDA_ARCHITECTURES': '120-real',
    'LSSO_MATHDX_LTO_ARCHITECTURES': '120',
    'LSSO_MATHDX_BUILD_DIR': str(MATHDX_CACHE / 'build-sm120'),
    'LSSO_MATHDX_RECONFIGURE': '1',
})
subprocess.run(['bash', 'tools/build_mathdx_backend.sh'], cwd=ROOT, env=build_env, check=True)
from lsso.mathdx_backend import is_mathdx_available, mathdx_load_error
assert is_mathdx_available(), f'MathDx backend failed to load: {mathdx_load_error()}'
print('MathDx backend: loaded (SM120 fused kernels enabled)')

In [ ]:
# Cache one complete shard per split, then run two real train/validation batches.
for filename in ('imagenet1k-train-0000.tar', 'imagenet1k-validation-00.tar'):
    subprocess.run([sys.executable, 'tools/hf_wds_stream.py', '--repo',
                    'timm/imagenet-1k-wds', '--filename', filename,
                    '--cache-dir', CONFIG['cache_dir'], '--quiet'],
                   check=True, env=os.environ.copy())
smoke_output = str(Path(CONFIG['output']).with_name(Path(CONFIG['output']).name + '_smoke'))
smoke = [sys.executable, 'experiments/imagenet_wds_train.py',
         '--model', CONFIG['model'], '--cache-dir', CONFIG['cache_dir'],
         '--output', smoke_output, '--epochs', '1', '--steps-per-epoch', '2',
         '--max-val-steps', '2', '--batch-size', '8', '--eval-batch-size', '8',
         '--workers', '0', '--eval-workers', '0', '--shard-limit', '1', '--shuffle-buffer', '128',
         '--seed', str(CONFIG['seed']), '--require-mathdx', '--no-resume']
subprocess.run(smoke, check=True, env=os.environ.copy())

## 4. Train with a foreground heartbeat (or resume)
Keep this cell running for the entire job. It refreshes GPU, cache, metric, and log status every two minutes so Colab sees an active foreground cell. `last.pt`, `best.pt`, `metrics.csv`, and `train.log` are written under `CONFIG['output']`; after a real runtime interruption, reconnect and run Sections 0-4 again to resume from `last.pt`.

In [ ]:
pid_path = Path(CONFIG['output']) / 'trainer.pid'
if pid_path.exists():
    old_pid = int(pid_path.read_text())
    old_stat = Path(f'/proc/{old_pid}/stat')
    if old_stat.exists() and old_stat.read_text().split()[2] != 'Z':
        raise RuntimeError(f'trainer PID {old_pid} is already running; do not launch twice')
command = [sys.executable, '-u', 'experiments/imagenet_wds_train.py',
           '--model', CONFIG['model'], '--cache-dir', CONFIG['cache_dir'],
           '--output', CONFIG['output'], '--epochs', str(CONFIG['epochs']),
           '--batch-size', str(CONFIG['batch_size']),
           '--eval-batch-size', str(CONFIG['eval_batch_size']),
           '--grad-accum', str(CONFIG['grad_accum']),
           '--workers', str(CONFIG['workers']),
           '--eval-workers', str(CONFIG['eval_workers']),
           '--seed', str(CONFIG['seed']), '--require-mathdx', '--resume']
from IPython.display import clear_output
output_dir = Path(CONFIG['output'])
output_dir.mkdir(parents=True, exist_ok=True)
log_path = output_dir / 'train.log'
metrics_path = output_dir / 'metrics.csv'
log = log_path.open('a', buffering=1)
process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT,
                           env=os.environ.copy())
pid_path.write_text(str(process.pid))
try:
    while process.poll() is None:
        clear_output(wait=True)
        print(time.strftime('%Y-%m-%d %H:%M:%S'),
              'pid=', process.pid, 'state=running')
        subprocess.run([
            'nvidia-smi',
            '--query-gpu=name,utilization.gpu,memory.used,memory.total,temperature.gpu',
            '--format=csv,noheader'], check=False)
        print('cached shards:', len(list(Path(CONFIG['cache_dir']).glob('*.tar'))))
        if metrics_path.exists():
            print('--- metrics ---')
            print('\n'.join(metrics_path.read_text(errors='replace').splitlines()[-6:]))
        if log_path.exists():
            print('--- log tail ---')
            print('\n'.join(log_path.read_text(errors='replace').splitlines()[-20:]))
        print('next refresh in 120 seconds')
        for _ in range(24):
            if process.poll() is not None:
                break
            time.sleep(5)
    returncode = process.wait()
    clear_output(wait=True)
    print('trainer exited with return code', returncode)
    print('\n'.join(log_path.read_text(errors='replace').splitlines()[-40:]))
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, command)
except KeyboardInterrupt:
    process.terminate()
    process.wait(timeout=30)
    raise
finally:
    log.close()
    if pid_path.exists() and pid_path.read_text().strip() == str(process.pid):
        pid_path.unlink()

In [ ]:
# Run this after the foreground training cell exits or after reconnecting.
pid_path = Path(CONFIG['output']) / 'trainer.pid'
if pid_path.exists():
    pid = int(pid_path.read_text())
    stat_path = Path(f'/proc/{pid}/stat')
    state = 'running' if stat_path.exists() and stat_path.read_text().split()[2] != 'Z' else 'stopped'
else:
    state = 'not running'
print('state:', state)
log_path = Path(CONFIG['output']) / 'train.log'
if log_path.exists():
    print('--- log tail ---')
    print('\n'.join(log_path.read_text(errors='replace').splitlines()[-15:]))
if (Path(CONFIG['output']) / 'metrics.csv').exists():
    print((Path(CONFIG['output']) / 'metrics.csv').read_text().splitlines()[-5:])
print('cached shards:', len(list(Path(CONFIG['cache_dir']).glob('*.tar'))))

## 5. Paired experiment
After RRLSSO finishes, change `model` and `output` to `vision_llama_base_lsso_r32` and rerun Sections 3–4. Both runs reuse exactly the same cached ImageNet shards and augmentation recipe. The registered MHA entry is an official-compatible reproduction path, not a required new formal run.